# Merging Datasets

In this notebook, we will focus on merging the two datasets together into different dataframes in order to help with buidling our visualizations.

In [1]:
import pandas as pd

### Initialize helper code and read filtered data into data frames

In [2]:
%run helpers.ipynb
imdb_df, oscar_df = read_data('imdb_filtered.csv', 'oscar_filtered.csv')

## Joining Oscar data with IMDb data

We want to begin joining the Oscar dataset with the IMDb dataset, using `FilmId` and `id` as keys. These columns both contain the IMDb Id of each film.

It is important to note that this will be done using a left join, as we want to add additional information from the IMDb dataset to the list of Oscar nominees in order to show more details about each film.

The **id** column from the IMDb dataset will be dropped as we don't want duplicate columns. The unamed columns from joining can also be dropped since they only contain indexes.

In [9]:
# join datasets together by film id
# imdb is id; oscar is FilmId
oscar_with_imdb = pd.merge(oscar_df, imdb_df, left_on='FilmId', right_on='id', how='left', indicator=True)
oscar_with_imdb = oscar_with_imdb.drop(columns=['Unnamed: 0_x', 'Unnamed: 0_y', 'id'])
oscar_with_imdb.head()


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,Winner,...,budget,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages,_merge
0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,NaN,...,NaN,NaN,NaN,NaN,1928-01-29,['United States'],['First National Pictures'],['Drama'],"['None', 'English']",both
1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,NaN,...,NaN,NaN,NaN,NaN,1927-09-01,['United States'],['First National Pictures'],"['Boxing', 'Drama', 'Romance', 'Sport', 'War']","['None', 'English']",both
2,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command,tt0019071,Emil Jannings,Emil Jannings,True,...,NaN,NaN,NaN,NaN,1928-01-21,['United States'],['Paramount Pictures'],"['Political Drama', 'Showbiz Drama', 'Tragedy'...","['None', 'English']",both
3,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,True,...,NaN,NaN,NaN,"$859,900",1927-10-01,['United States'],['Paramount Pictures'],['Drama'],"['None', 'English']",both
4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,NaN,...,NaN,NaN,NaN,NaN,1928-06-04,['United States'],['DeMille Pictures Corporation'],['Drama'],['None'],both


### Join Failures
There are now 27 columns in this merged data frame. There are 10 columns from the Oscar dataset and 17 from the IMDb dataset. However, we want to know how much data was not joined. There may be films in the list of Oscar nominees that are missing from the IMDb dataset.

In [4]:
# here we want to see a list of join failures, i.e. Oscar nominees missing from the imdb dataset
not_joined = (
    oscar_with_imdb
    .loc[oscar_with_imdb['_merge'] == 'left_only']
    .groupby('Category')
    .size()
    .sort_values(ascending=False)
)

not_joined

Category
FOREIGN LANGUAGE FILM                                  21
MUSIC (Original Song)                                   8
CINEMATOGRAPHY                                          5
COSTUME DESIGN                                          3
MUSIC (Song--Original for the Picture)                  3
MUSIC (Scoring of a Musical Picture)                    3
ANIMATED FEATURE FILM                                   3
MUSIC (Song)                                            3
SPECIAL AWARD                                           3
WRITING (Adapted Screenplay)                            2
DIRECTING                                               2
INTERNATIONAL FEATURE FILM                              2
ART DIRECTION                                           2
ACTRESS IN A SUPPORTING ROLE                            2
ACTOR                                                   1
HONORARY AWARD                                          1
FILM EDITING                                            1
BEST 

In [5]:
# total join failures
print(not_joined.sum())

79


We can see that of the 9,000+ rows in the Oscar dataset, 79 of them failed to appear in the IMDb dataset. The highest category was for Foreign Language Film. Overall, we can consider this number of missing rows in the merged dataset to be fine since it is relatively low. Additionally, most of the categories listed are missing only 1-5 films, which should not skew our dataset.

One possible reason for these join failures could be related to the IMDb dataset containing a list of the Top 500-600 films from the years 1920-2025. Therefore, it could be that these 79 films were not voted highly on IMDb to have been included in the raw dataset.



## Dropping Oscar nominees from IMDb data

Another dataframe that will be useful in our investigation is the list of films in our IMDb dataset that were **not** nominated for any Oscar awards. This is particularly important for helping us investigate audience popularity against award/prestige bias.

In [6]:
# create a dataframe with movies in imdb dataset that were not nominated for oscars
imdb_non_oscar = imdb_df[~imdb_df['id'].isin(oscar_df['FilmId'])]
imdb_non_oscar.head()

,Unnamed: 0,id,title,rating,votes,meta_score,writers,directors,stars,budget,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages
0,0,tt0011370,Klostret i Sendomir,6.9,485,NaN,"['Franz Grillparzer', 'Victor Sjöström']",['Victor Sjöström'],"['Tore Svennberg', 'Tora Teje', 'Richard Lund'...",NaN,NaN,NaN,NaN,1920-01-01,['Sweden'],['Svenska Biografteatern AB'],['Drama'],['None']
1,1,tt0011413,The Lost City,4.7,35,NaN,['Frederick Chapin'],['E.A. Martin'],"['Juanita Hansen', 'George Chesebro', 'Frank C...",NaN,NaN,NaN,NaN,1920-01-01,['United States'],['Selig Polyscope Company'],"['Action', 'Adventure']","['None', 'English']"
2,2,tt0276209,Hypnose,7.0,19,NaN,['Karl Schneider'],['Richard Eichberg'],"['Lee Parry', 'Gertrud de Lalsky', 'Karl Halde...",NaN,NaN,NaN,NaN,1920-01-03,['Germany'],['Richard Eichberg-Film GmbH'],"['Drama', 'Mystery']",['None']
3,3,tt0010495,My Husband's Other Wife,5.3,17,NaN,['Stanley Olmstead'],['J. Stuart Blackton'],"['Sylvia Breamer', 'Robert Gordon', 'Warren Ch...",NaN,NaN,NaN,NaN,1920-01-04,['United States'],['J. Stuart Blackton Feature Pictures'],"['Drama', 'Romance']",['None']
4,4,tt0010502,Nachtgestalten,5.6,25,NaN,"['Richard Oswald', 'Karl Hans Strobl']",['Richard Oswald'],"['Paul Wegener', 'Reinhold Schünzel', 'Erna Mo...",NaN,NaN,NaN,NaN,1920-01-09,['Germany'],['Richard-Oswald-Produktion'],['Horror'],"['None', 'German']"


We have managed to filter out around 5,000 films from our IMDb dataset as seen by the shape below. Note, the number of rows filtered won't be equal to the shape of the Oscar dataset (9,073 rows) because the same film can be nominated multiple times for an Oscar.

In [14]:
print("Shape of IMDb dataset with 0 Oscar nominations: ", imdb_non_oscar.shape)

Shape of IMDb dataset with 0 Oscar nominations:  (55684, 18)


In [8]:
write_joined(oscar_with_imdb, imdb_non_oscar)